# Operational Control Tower Dashboard

This notebook consolidates analytical outputs generated throughout the repository into operational reporting views.

The objective is to simulate the type of morning operational review that inventory planners, supply chain analysts, and warehouse managers may perform using ERP-derived operational datasets.

The dashboard focuses on:

- inventory-control risks
- replenishment priorities
- supplier exposure
- warehouse automation performance
- warehouse-transition progress

The reporting layer translates analytical outputs into actionable operational recommendations.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(
        str(PROJECT_ROOT / "src")
    )

from dashboard_utils import (
    render_kpi_board,
    render_control_tower,
    render_priority_queue
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

ERP_DIR = Path("../outputs/erp_reporting")
TRANSITION_DIR = Path("../outputs/powerbi_transition")
POWERBI_DIR = Path("../outputs/powerbi_operational")


# Load Outputs

planner_kpi_dashboard = pd.read_csv(
    ERP_DIR / "planner_kpi_dashboard.csv"
)

planner_exception_queue = pd.read_csv(
    ERP_DIR / "planner_exception_queue.csv"
)

supplier_replenishment_exposure = pd.read_csv(
    ERP_DIR / "supplier_replenishment_exposure.csv"
)

warehouse_transition_kpis = pd.read_csv(
    TRANSITION_DIR / "warehouse_transition_kpis.csv"
)

warehouse_before_after_summary = pd.read_csv(
    TRANSITION_DIR / "warehouse_before_after_summary.csv"
)

kpi_summary = pd.read_csv(
    POWERBI_DIR / "kpi_summary.csv"
)

## Supply Chain Control Tower

The control tower consolidates key operational signals requiring management attention.

In [11]:
planner_exceptions = int(
    planner_kpi_dashboard.loc[
        planner_kpi_dashboard["Metric"] ==
        "Planner Exceptions",
        "Value"
    ].iloc[0]
)

expedite_actions = int(
    planner_kpi_dashboard.loc[
        planner_kpi_dashboard["Metric"] ==
        "Expedite Actions",
        "Value"
    ].iloc[0]
)

open_requisitions = int(
    planner_kpi_dashboard.loc[
        planner_kpi_dashboard["Metric"] ==
        "Open Requisitions",
        "Value"
    ].iloc[0]
)

supplier_reviews = (
    supplier_replenishment_exposure["Supplier_Risk"]
    .eq("Review")
    .sum()
)

movement_migration = float(
    warehouse_transition_kpis.loc[
        warehouse_transition_kpis["Transition_KPI"] ==
        "Movement migration %",
        "Value"
    ].iloc[0]
)



def inventory_status(exceptions):

    if exceptions >= 40:
        return "Red"

    if exceptions >= 20:
        return "Amber"

    return "Green"


def expedite_status(expedites):

    if expedites >= 2:
        return "Red"

    if expedites >= 1:
        return "Amber"

    return "Green"


def supplier_status(reviews):

    if reviews >= 3:
        return "Red"

    if reviews >= 1:
        return "Amber"

    return "Green"


control_tower = pd.DataFrame({
    "Control": [
        "Inventory Risk",
        "Replenishment Urgency",
        "Supplier Exposure",
        "Warehouse Transition"
    ],

    "Status": [
        inventory_status(planner_exceptions),
        expedite_status(expedite_actions),
        supplier_status(supplier_reviews),
        "Green"
    ],

    "Operational_Reading": [
        f"{planner_exceptions} planner exceptions identified",
        f"{expedite_actions} materials require expedited supply",
        f"{supplier_reviews} suppliers flagged for review",
        f"{movement_migration:.1f}% movement migration achieved"
    ]
})

top_material = (
    planner_exception_queue
    .sort_values(
        "Priority_Score",
        ascending=False
    )
    .iloc[0]
)

top_supplier = (
    supplier_replenishment_exposure[
        supplier_replenishment_exposure["Supplier_Risk"]
        == "Review"
    ]
    .sort_values(
        "Requisition_Value_EUR",
        ascending=False
    )
    .iloc[0]
)

render_control_tower(
    title="Supply Chain Control Tower",
    dashboard_df=control_tower,
    actions=[
        f"Review {planner_exceptions} inventory exceptions",
        f"Expedite {expedite_actions} safety-stock breaches",
        f"Review supplier {top_supplier['Supplier_ID']}",
        f"Prioritise {top_material['Part_Name']}"
    ],
    legend=[
        (
            "🔴",
            "Immediate operational attention required."
        ),
        (
            "🟠",
            "Monitor and review during planning cycle."
        ),
        (
            "🟢",
            "Operating within expected thresholds."
        )
    ]
)

Control,Status,Operational Reading
Inventory Risk,,47 planner exceptions identified
Replenishment Urgency,,2 materials require expedited supply
Supplier Exposure,,3 suppliers flagged for review
Warehouse Transition,,84.4% movement migration achieved


## Inventory Planner Queue

The planner queue highlights the highest-priority inventory positions requiring replenishment review.

In [3]:
planner_priority_queue = (
    planner_exception_queue
    .sort_values(
        "Priority_Score",
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)

planner_priority_queue_display = pd.DataFrame({
    "Priority":
        range(
            1,
            len(planner_priority_queue)+1
        ),

    "Item":
        planner_priority_queue["Part_Name"],

    "Criticality":
        planner_priority_queue["Criticality"],

    "Action":
        planner_priority_queue[
            "Suggested_Planner_Action"
        ]
})

render_priority_queue(
    "Top Priority Materials",
    planner_priority_queue_display,
    legend=[
        (
            "Expedite Supply",
            "Inventory position below safety-stock protection."
        ),
        (
            "Create Requisition",
            "Inventory position below reorder point."
        ),
        (
            "Priority Score",
            "Ranking generated from inventory criticality and replenishment risk."
        )
    ]
)

Priority,Item,Criticality,Action
1,ERCO Lift O-Ring Kit,High,Create Requisition
2,A2024 LL Air Filter,Medium,Create Requisition
3,Scissor Lift Air Filter,Medium,Create Requisition
4,Exact Precision Lubrication Kit,Medium,Create Requisition
5,Radar Calibration Lubrication Kit,High,Create Requisition
6,Eyelight Rubber Pad,Medium,Create Requisition
7,A2024 LL Protective Cover,Medium,Create Requisition
8,EM 9000 Rubber Pad,Critical,Create Requisition
9,A2024 LL I/O Board,Critical,Create Requisition
10,Alignment Lift Service Seal Pack,Critical,Create Requisition


## Supplier Review Queue

Suppliers flagged for review require monitoring due to lead-time exposure or reduced reliability.

In [4]:
supplier_review_queue = (
    supplier_replenishment_exposure[
        supplier_replenishment_exposure[
            "Supplier_Risk"
        ] == "Review"
    ]
    .sort_values(
        "Requisition_Value_EUR",
        ascending=False
    )
)



supplier_review_display = pd.DataFrame({
    "Priority":
        range(
            1,
            len(supplier_review_queue) + 1
        ),

    "Item":
        supplier_review_queue["Supplier_Name"],

    "Criticality":
        ["High"] * len(
            supplier_review_queue
        ),

    "Action":
        "Review Supplier Performance"
})



render_priority_queue(
    "Supplier Review Queue",
    supplier_review_display
)

Priority,Item,Criticality,Action
1,EU Industrial Components 18,High,Review Supplier Performance
2,Eastern Europe Industrial Components 05,High,Review Supplier Performance
3,Eastern Europe Industrial Components 14,High,Review Supplier Performance


## Inventory Service Level Monitoring

Inventory planners must ensure that inventory positions remain capable of supporting target service levels.

This dashboard highlights materials whose current inventory positions may place service-level performance at risk and identifies inventory positions requiring monitoring or replenishment intervention.

In [5]:
service_level_dashboard = planner_exception_queue.copy()

def service_risk(row):

    if row["Available_Qty"] < row["Safety_Stock_Qty"]:
        return "Below Target"

    if row["Available_Qty"] < (
        row["Safety_Stock_Qty"] * 1.25
    ):
        return "At Risk"

    return "On Target"


service_level_dashboard["Service_Level_Status"] = (
    service_level_dashboard.apply(
        service_risk,
        axis=1
    )
)

service_level_summary = (
    service_level_dashboard
    .groupby("Service_Level_Status")
    .agg(
        Parts=("Part_ID", "count")
    )
    .reset_index()
)

service_level_review = (
    service_level_dashboard[
        service_level_dashboard[
            "Service_Level_Status"
        ] != "On Target"
    ]
    .sort_values(
        "Priority_Score",
        ascending=False
    )
    .head(10)
)

service_level_display = pd.DataFrame({
    "Priority":
        range(
            1,
            len(service_level_review)+1
        ),

    "Item":
        service_level_review["Part_Name"],

    "Criticality":
        service_level_review["Criticality"],

    "Available":
        service_level_review["Available_Qty"],

    "Safety Stock":
        service_level_review["Safety_Stock_Qty"],

    "Coverage (weeks)":
        service_level_review[
            "Stock_Coverage_Weeks"
        ].round(1),

    "Action":
        service_level_review[
            "Service_Level_Status"
        ]
})

render_priority_queue(
    "Service Level Monitoring",
    service_level_display,
    legend=[
        (
            "Below Target",
            "Inventory position below safety-stock protection."
        ),
        (
            "At Risk",
            "Inventory position approaching safety-stock threshold."
        ),
        (
            "Criticality",
            "Operational importance of the material and its potential impact on service delivery if unavailable."
        ),
        (
            "Coverage",
            "Estimated weeks of inventory remaining."
        )
    ]
)

Priority,Item,Criticality,Available,Safety Stock,Coverage (weeks),Action
1,Scissor Lift Air Filter,Medium,16,13,4.2,At Risk
2,Radar Calibration Lubrication Kit,High,11,11,11.1,At Risk
3,A2024 LL Protective Cover,Medium,12,10,26.7,At Risk
4,Brake Tester Spindle Bearing,Medium,6,5,31.2,At Risk
5,ERCO Lift Filter Kit,High,14,13,10.7,At Risk
6,Alignment Lift Service Seal Pack,Critical,6,5,52.0,At Risk
7,A2024 LL I/O Board,Critical,1,1,nan,At Risk
8,HD1200 Truck Protective Cover,Medium,6,5,5.4,At Risk
9,Camera Calibration Pneumatic Cylinder,High,2,2,156.0,At Risk
10,Basic Line TC Position Sensor,High,2,2,nan,At Risk


## Warehouse Transition Status

The warehouse-transition programme continues to deliver strong movement migration while reducing manual handling requirements.

In [6]:
transition_control = pd.DataFrame({
    "Control": [
        "SKU Migration",
        "Movement Migration",
        "Inventory Value Migration",
        "Residual Manual Activity"
    ],

    "Status": [
        "Amber",
        "Green",
        "Amber",
        "Green"
    ],

    "Operational_Reading": [
        "37.7% of SKUs migrated",
        "84.4% of movements migrated",
        "25.9% of inventory value migrated",
        "15.6% residual manual movement activity"
    ]
})




render_control_tower(
    title="Warehouse Transition Status",
    dashboard_df=transition_control,
    actions=[
        "Continue migration of high-movement SKUs",
        "Monitor residual manual warehouse activity",
        "Review future automation opportunities"
    ]
)

Control,Status,Operational Reading
SKU Migration,,37.7% of SKUs migrated
Movement Migration,,84.4% of movements migrated
Inventory Value Migration,,25.9% of inventory value migrated
Residual Manual Activity,,15.6% residual manual movement activity


## Operational Summary

The dashboards demonstrate how ERP-derived operational datasets can be transformed into decision-support tools supporting inventory planning, replenishment management, supplier review, warehouse automation, and warehouse-transition monitoring.

The reporting approach focuses on operational priorities rather than descriptive reporting, enabling planners and managers to identify risks, prioritise actions, and monitor progress using a structured control-tower framework.

## Export the complete dashboard

In [19]:
control_tower_html=render_control_tower(
    title="Supply Chain Control Tower",
    dashboard_df=control_tower,
    actions=[
        f"Review {planner_exceptions} inventory exceptions",
        f"Expedite {expedite_actions} safety-stock breaches",
        f"Review supplier {top_supplier['Supplier_ID']}",
        f"Prioritise {top_material['Part_Name']}"
    ],
    legend=[
        (
            "🔴",
            "Immediate operational attention required."
        ),
        (
            "🟠",
            "Monitor and review during planning cycle."
        ),
        (
            "🟢",
            "Operating within expected thresholds."
        )
    ],
    return_html=True
)

priority_materials_html=render_priority_queue(
    "Top Priority Materials",
    planner_priority_queue_display,
    legend=[
        (
            "Expedite Supply",
            "Inventory position below safety-stock protection."
        ),
        (
            "Create Requisition",
            "Inventory position below reorder point."
        ),
        (
            "Priority Score",
            "Ranking generated from inventory criticality and replenishment risk."
        )
    ],
    return_html=True
)

supplier_review_html=render_priority_queue(
    "Supplier Review Queue",
    supplier_review_display,
    return_html=True
)

service_level_html=render_priority_queue(
    "Service Level Monitoring",
    service_level_display,
    legend=[
        (
            "Below Target",
            "Inventory position below safety-stock protection."
        ),
        (
            "At Risk",
            "Inventory position approaching safety-stock threshold."
        ),
        (
            "Criticality",
            "Operational importance of the material and its potential impact on service delivery if unavailable."
        ),
        (
            "Coverage",
            "Estimated weeks of inventory remaining."
        )
    ],
    return_html=True
)


warehouse_transition_html=render_control_tower(
    title="Warehouse Transition Status",
    dashboard_df=transition_control,
    actions=[
        "Continue migration of high-movement SKUs",
        "Monitor residual manual warehouse activity",
        "Review future automation opportunities"
    ],
    return_html=True
)

In [20]:
report_html = f"""
<html>

<head>

<title>
Supply Chain Control Tower
</title>

<style>

body {{
    background:#0f1117;
    font-family:Segoe UI;
    margin:40px;
}}

</style>

</head>

<body>

<h1 style="color:white;">
Supply Chain Operational Dashboard
</h1>

{control_tower_html}

<br><br>

{priority_materials_html}

<br><br>

{supplier_review_html}

<br><br>

{service_level_html}

<br><br>

{warehouse_transition_html}

</body>

</html>
"""

In [21]:
REPORT_DIR = Path(
    "../outputs/reports"
)

REPORT_DIR.mkdir(
    exist_ok=True
)

with open(
    REPORT_DIR /
    "supply_chain_operational_dashboard.html",
    "w",
    encoding="utf-8"
) as f:

    f.write(report_html)

In [22]:
import webbrowser

webbrowser.open(
    str(
        (
            REPORT_DIR /
            "supply_chain_operational_dashboard.html"
        ).resolve()
    )
)

True